# Module 2b — Long context vs RAG, measured

**The problem.** Context windows are huge now, so a tempting reflex is: skip retrieval, paste the whole corpus into the prompt. This notebook *measures* where that reflex fails.

1. **Silent truncation** — a prompt past the window is cut without an error; the model answers from what survived and sounds confident. (Measured here.)
2. **Position sensitivity ('lost in the middle')** — retrieval reliability across a long context is not guaranteed uniform; a buried fact can be retrieved worse than one at the edge. This is a documented effect (Liu et al. 2023); here we *test for it* on one 7B at one length and report what we find.
3. **Cost and latency** scale with every token you stuff; RAG pays a small retrieval cost to send few. (Measured here.)

Needs Ollama (`qwen2.5:7b`, `bge-m3`) and, for the independent token count, the `transformers` Qwen2.5 tokenizer (cached; used offline). The long-context token counts (Sections 1-2) are cross-checked two ways: the model's `prompt_eval_count` and the tokenizer.


## 0. The honest instrument: read what the model actually saw

Ollama returns `prompt_eval_count` (prompt tokens processed). If a prompt overflows `num_ctx`, Ollama **silently drops the middle**: it keeps the first `num_keep` tokens (default 4) and the last half of the window, so the count lands at `num_keep + (num_ctx - num_keep)//2` — *below* `num_ctx`, not pinned to it. (`num_keep` defaults to 4; we *observe* this floor at `num_ctx=2048` in Section 1, and take the general form as inferred from it — other runtimes may truncate differently.) We also count tokens independently with the Qwen tokenizer, so we can *prove* a prompt fit (or didn't).


In [1]:
import sys, pathlib, importlib, time, os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
here = pathlib.Path.cwd()
for p in (here, here/"notebooks", here.parent):
    if (p/"arch_kit.py").exists(): sys.path.insert(0, str(p)); break
import arch_kit; importlib.reload(arch_kit)
from arch_kit import embed, cosine

# --- knobs (edit and re-run) ---
SMALL_CTX    = 2048     # truncating window for the silent-truncation demo
BIG_CTX      = 16384    # window large enough for the same long prompt
POSITION_CTX = 16384    # window for the position sweep
N_DISTRACT   = 400      # lookalike lines (~9.6k tokens) -> fits POSITION_CTX with headroom

def ask(prompt, system, num_ctx, num_predict=30):
    """Return (text, prompt_tokens, gen_tokens, seconds) straight from Ollama."""
    msgs = ([{"role":"system","content":system}] if system else []) + [{"role":"user","content":prompt}]
    t0 = time.time()
    r = arch_kit._post("/api/chat", {"model": arch_kit.CHAT_MODEL, "messages": msgs, "stream": False,
         "options": {"temperature": 0.0, "num_predict": num_predict, "num_ctx": num_ctx}})
    return (r.get("message",{}).get("content","").strip(),
            r.get("prompt_eval_count"), r.get("eval_count"), round(time.time()-t0,1))

NUM_KEEP = 4
def trunc_floor(nc): return NUM_KEEP + (nc - NUM_KEEP)//2   # tokens Ollama keeps when a prompt overflows nc

# independent token count (same BPE vocab as the 7B); falls back gracefully if the tokenizer isn't cached
try:
    from transformers import AutoTokenizer
    _TOK = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
    def count_tokens(system, user):
        msgs=[{"role":"system","content":system},{"role":"user","content":user}]
        return len(_TOK(_TOK.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False))["input_ids"])
    HAVE_TOK = True
except Exception as e:
    HAVE_TOK = False
    def count_tokens(system, user): return None
    print("note: Qwen tokenizer unavailable, exact counts skipped ->", e)

print("instrument ready — qwen2.5:7b via", arch_kit.OLLAMA, "| independent tokenizer:", HAVE_TOK)
print("truncation floor: num_ctx=2048 ->", trunc_floor(2048), "| num_ctx=16384 ->", trunc_floor(16384))


instrument ready — qwen2.5:7b via http://127.0.0.1:11434 | independent tokenizer: True
truncation floor: num_ctx=2048 -> 1026 | num_ctx=16384 -> 8194


## 1. Silent truncation — the same long prompt, two windows

One needle sentence in the middle of a long filler document. We send the identical prompt twice: once with a window too small for it, once large enough. Watch `prompt_tokens`, and compare it to the independent count of what we actually sent.


In [2]:
NEEDLE = "The access code for vault seven is PURPLE-GIRAFFE-42."
filler = [f"Record line {i}: the quarterly logistics review noted routine throughput, no incidents."
          for i in range(900)]
mid = len(filler)//2
long_ctx = "\n".join(filler[:mid] + [NEEDLE] + filler[mid:])
SYS = "Answer only from the context.\n\nCONTEXT:\n" + long_ctx
Q   = "What is the access code for vault seven? Answer with only the code."

sent = count_tokens(SYS, Q)
print(f"independent count of what we SENT: {sent} tokens\n" if sent else "independent count unavailable\n")

res = {}
for nc in (SMALL_CTX, BIG_CTX):
    txt, pt, gt, sec = ask(Q, SYS, num_ctx=nc)
    res[nc] = pt
    hit = "PURPLE-GIRAFFE-42" in txt.upper()
    if pt == trunc_floor(nc):       tag = "(== truncation floor -> CUT)"
    elif sent and pt == sent:       tag = "(== what we sent -> held)"
    else:                           tag = "(partial)"
    print(f"num_ctx={nc:6}  prompt_tokens={pt:6} {tag:28}  found={hit!s:5}  {sec:6}s  reply={txt[:28]!r}")
FULL_TOKENS, SMALL_TOKENS = res[BIG_CTX], res[SMALL_CTX]


independent count of what we SENT: 16143 tokens



num_ctx=  2048  prompt_tokens=  1026 (== truncation floor -> CUT)  found=False     9.8s  reply='42715893'


num_ctx= 16384  prompt_tokens= 16143 (== what we sent -> held)     found=True    103.2s  reply='PURPLE-GIRAFFE-42'


**Read it.** The same prompt went in both times. At the large window `prompt_tokens` equals the independent count of what we sent — the whole prompt was processed, and the needle is found. At the small window `prompt_tokens` collapses to the **truncation floor** (`num_keep + (num_ctx-num_keep)//2`): Ollama kept the first few tokens and the last half of the window and **silently dropped the middle** — where the needle sat — so the model answered with a confident, wrong code. **No exception was raised.** The signal of truncation is `prompt_tokens` landing at that floor, well below what you sent — which is exactly why you compare it to an independent count of your own input.


## 2. Does position hurt? A sweep that is PROVEN to fit its window

The Section 1 trap is easy to fall into here: if the sweep prompt overflows the window, Ollama keeps the tail, so the needle is 'found' only when it sits near the end — a **fake position effect that is really truncation**. We avoid it by proving the prompt fits: the independent token count must be below `POSITION_CTX`, and every run's `prompt_tokens` must equal that count (not the truncation floor). Then we move the needle through ~400 lookalike distractors and read the result honestly.


In [3]:
DISTRACT = [f"Entry {i:04d}: the access code for vault {i} is CODE-{i:04d}." for i in range(N_DISTRACT) if i != 7]
def at_position(frac):
    k = int(len(DISTRACT)*frac)
    return "\n".join(DISTRACT[:k] + [NEEDLE] + DISTRACT[k:])

sysmid = "Answer only from the context.\n\nCONTEXT:\n" + at_position(0.5)
EXACT = count_tokens(sysmid, Q)
print(f"independent count of the sweep prompt: {EXACT} tokens  (window {POSITION_CTX}, floor {trunc_floor(POSITION_CTX)})")
assert EXACT is None or EXACT < POSITION_CTX, "sweep prompt overflows the window -> would truncate; lower N_DISTRACT"

rows=[]; SWEEP_PTS=[]
for frac in (0.0, 0.25, 0.5, 0.75, 1.0):
    txt, pt, gt, sec = ask(Q, "Answer only from the context.\n\nCONTEXT:\n"+at_position(frac), num_ctx=POSITION_CTX)
    SWEEP_PTS.append(pt)
    held = (pt != trunc_floor(POSITION_CTX)) and (EXACT is None or pt == EXACT)
    rows.append((f"{frac:.0%}", pt, held, "PURPLE-GIRAFFE-42" in txt.upper(), f"{sec}s"))
import pandas as pd
df = pd.DataFrame(rows, columns=["needle position","prompt_tokens","whole doc held","found","time"])
print(df.to_string(index=False))
print("\n(the 'time' column varies with KV-cache prefix reuse across runs, not with position -- the token")
print(" counts are identical; don't read the timing as a position effect.)")


independent count of the sweep prompt: 9521 tokens  (window 16384, floor 8194)


needle position  prompt_tokens  whole doc held  found  time
             0%           9521            True   True 53.8s
            25%           9521            True   True 57.4s
            50%           9521            True   True 45.4s
            75%           9521            True   True 30.9s
           100%           9521            True   True 16.7s

(the 'time' column varies with KV-cache prefix reuse across runs, not with position -- the token
 counts are identical; don't read the timing as a position effect.)


**Read it.** First the guard: `whole doc held` is True at every row and `prompt_tokens` equals the independent count (9,521) — nothing was truncated, so any difference in `found` is genuinely *position*. **The result in this run is flat: the needle is found at all five positions.** So at ~9.5k tokens, proven to fit, this 7B is **robust to position** — the documented 'lost in the middle' effect (Liu et al. 2023) did **not** bite here. That is the honest result, not a failed demo: the effect strengthens with longer contexts, more distractors, and varies by model, so a flat result at this length is expected for a strong model. (A miss at some positions would be the effect, live.) **The architectural point holds either way:** you don't control where a fact lands when you stuff a corpus, so don't *assume* uniform retrieval — measure it. And note what the guard prevented: without the token check, an overflowing sweep shows a crisp fake 'end bias' that is pure truncation — which is exactly the trap an earlier draft of this notebook fell into.


## 3. RAG vs stuff — head to head on tokens, latency, and the answer

A small corpus and a query. **Stuff:** paste every document. **RAG:** embed the query, keep top-k by cosine, send only those. We measure prompt tokens and latency — counting the retrieval cost honestly.


In [4]:
DOCS = [
    "The company was founded in 1998 in Basel by two power-grid engineers.",
    "Annual leave is 25 days; carry-over of up to 5 days is allowed until March.",
    "The incident hotline is staffed 24/7 and the escalation SLA is 15 minutes.",
    "Expense claims over 500 CHF require a second approver in the finance tool.",
    "The data-retention policy keeps access logs for 18 months, then purges them.",
    "Parking at the Dreispitz site is by permit only; apply through facilities.",
    "The VPN disconnects idle sessions after 30 minutes for security.",
    "New hires complete the security onboarding within their first two weeks.",
]
GOLD_DOC = 4  # retention policy
QUERY = "How long are access logs kept before they are deleted?"

stuff_ctx = "\n".join(f"[{i}] {d}" for i,d in enumerate(DOCS))
s_txt, s_pt, _, s_sec = ask(QUERY, "Answer only from the context.\n\nCONTEXT:\n"+stuff_ctx, num_ctx=4096, num_predict=60)

t0 = time.time()
qv = embed(QUERY); sims = [(cosine(qv, embed(d)), i) for i,d in enumerate(DOCS)]
top = [i for _,i in sorted(sims, reverse=True)[:3]]
retrieval_sec = round(time.time()-t0, 1)   # the retrieval cost, counted (not hidden)
rag_ctx = "\n".join(f"[{i}] {DOCS[i]}" for i in top)
r_txt, r_pt, _, r_gen_sec = ask(QUERY, "Answer only from the context.\n\nCONTEXT:\n"+rag_ctx, num_ctx=4096, num_predict=60)

print(f"gold doc #{GOLD_DOC} in RAG top-3? {GOLD_DOC in top}   (retrieved: {top})\n")
print(f"STUFF  prompt_tokens={s_pt:4}  chat {s_sec:4}s                         ans={s_txt[:60]!r}")
print(f"RAG    prompt_tokens={r_pt:4}  chat {r_gen_sec:4}s + retrieval {retrieval_sec:4}s = {r_gen_sec+retrieval_sec:.1f}s  ans={r_txt[:60]!r}")
print(f"\nRAG sent {100*(1-r_pt/s_pt):.0f}% fewer prompt tokens. ('chat' is the full round-trip, not just generation.)")
print("NB: here we embed all 8 docs per query to keep the demo self-contained; a real system embeds once at")
print("INGEST, so per-query retrieval is a cheap lookup and the end-to-end latency picture flips to RAG's favour.")


gold doc #4 in RAG top-3? True   (retrieved: [4, 6, 1])

STUFF  prompt_tokens= 198  chat  5.4s                         ans='Access logs are kept for 18 months before they are deleted.'
RAG    prompt_tokens=  95  chat  2.0s + retrieval  2.6s = 4.6s  ans='Access logs are kept for 18 months before they are deleted.'

RAG sent 52% fewer prompt tokens. ('chat' is the full round-trip, not just generation.)
NB: here we embed all 8 docs per query to keep the demo self-contained; a real system embeds once at
INGEST, so per-query retrieval is a cheap lookup and the end-to-end latency picture flips to RAG's favour.


**Read it.** On eight short docs both answer correctly — stuffing is simpler and fine *here*, and note RAG is **not** faster end-to-end in this toy (re-embedding all 8 docs per query ≈ the token saving); the RAG win here is **tokens**, and at real scale **latency too**, because embeddings are computed once at ingest, not per query. The honest rule is about scale and placement: stuff when the whole corpus fits with headroom, nothing critical is buried, and the token bill is acceptable; **retrieve** when the corpus is larger than the window (or grows), when a buried fact could be missed, or when per-request cost/latency matter. A big window removes the hard *ceiling* (truncation); it does **not** remove the token bill, and it does not let you assume uniform retrieval. Long context and RAG are complementary: retrieve to shortlist, use the window to reason over the shortlist.


### The decision as a diagram

```mermaid
flowchart TD
    q[new question + corpus] --> fit{corpus fits the window<br/>with headroom?}
    fit -->|no| rag[RETRIEVE: embed, keep top-k, send a few]
    fit -->|yes| bury{could a key fact be buried<br/>away from where the model attends?}
    bury -->|yes| rag
    bury -->|no| cost{per-request cost/latency<br/>ok at peak scale?}
    cost -->|no| rag
    cost -->|yes| stuff[STUFF: paste the corpus]
    rag --> combo[often best: retrieve to shortlist,<br/>then use the window to reason over it]
```


## 4. Meaningful tests

Structural facts that can fail — never the model's (model/version-dependent) answer text.


In [5]:
def run_checks():
    checks=[]
    def check(n,c,w): checks.append((n,"PASS" if c else "FAIL",w))

    # truncation: the small window landed on the truncation floor, far below the full prompt
    check("small window hit the truncation floor (silent cut)",
          SMALL_TOKENS == trunc_floor(SMALL_CTX) and SMALL_TOKENS < FULL_TOKENS,
          "truncation shows as prompt_tokens at num_keep+half-window, below what was sent -- no error raised")
    check("large window processed EXACTLY what we sent (matches the independent count)",
          sent is not None and FULL_TOKENS == sent,
          "cross-check the model's prompt_tokens against your own tokenizer count -- they must match")

    # position sweep: PROVEN untruncated via the independent count, so 'found' is about position
    check("position sweep fits its window (independent count < num_ctx)",
          EXACT is not None and EXACT < POSITION_CTX,
          "if the sweep overflowed, a tail-survival pattern would masquerade as a position effect")
    check("every sweep run held the whole doc (no truncation floor, matches exact count)",
          all(pt != trunc_floor(POSITION_CTX) for pt in SWEEP_PTS)
          and (EXACT is None or all(pt == EXACT for pt in SWEEP_PTS)),
          "identical full-length prompt at every position -> any miss is position, not a silent cut")

    # RAG retrieves the gold doc and sends fewer tokens (sanity-level, but real)
    check("RAG retrieves the gold doc into top-k", GOLD_DOC in top,
          "if retrieval misses the answer, the RAG arm is testing nothing")
    check("RAG sends fewer prompt tokens than stuffing", r_pt < s_pt,
          "the whole point: pay a retrieval step, send a fraction of the tokens")

    import pandas as pd
    print(pd.DataFrame(checks, columns=["check","result","why it matters"]).to_string(index=False))
    assert all(r=="PASS" for _,r,_ in checks), "a long-context claim broke"
    print("\nall", len(checks), "meaningful checks passed")
run_checks()


                                                                        check result                                                                                    why it matters
                           small window hit the truncation floor (silent cut)   PASS truncation shows as prompt_tokens at num_keep+half-window, below what was sent -- no error raised
  large window processed EXACTLY what we sent (matches the independent count)   PASS         cross-check the model's prompt_tokens against your own tokenizer count -- they must match
                 position sweep fits its window (independent count < num_ctx)   PASS            if the sweep overflowed, a tail-survival pattern would masquerade as a position effect
every sweep run held the whole doc (no truncation floor, matches exact count)   PASS          identical full-length prompt at every position -> any miss is position, not a silent cut
                                        RAG retrieves the gold doc into top-k   PASS 

## Check your understanding

**Q1.** The small-window run returned a confident answer and raised no error. How would you *detect*, in production, that the model never saw half your prompt?
<details><summary>answer</summary>Compare the token accounting the provider returns (`prompt_eval_count` for Ollama; `usage.prompt_tokens` for hosted APIs) against a count of the input you actually sent. On Ollama a truncated prompt lands on the floor `num_keep + (num_ctx-num_keep)//2` — below `num_ctx`, not pinned to it — and well below your real input size. Never infer 'it worked' from a fluent answer: truncation is silent and the model fills the gap confidently.</details>

**Q2.** A bigger context window is advertised as making RAG unnecessary. Which problem does a bigger window actually solve, and which two does it not?
<details><summary>answer</summary>It raises the *truncation ceiling* — fewer prompts get cut. It does **not** guarantee uniform retrieval by position (the documented lost-in-the-middle effect, which depends on model and length), and it does **not** fix cost/latency (you pay for every stuffed token, every request). A big window removes a hard failure, not the economic or reliability case for retrieving first.</details>

**Q3.** On the eight-doc corpus, stuffing worked fine. Why is 'stuffing works' not a safe general conclusion?
<details><summary>answer</summary>Because it was measured on a corpus that fits with room to spare and has nothing buried. The failure modes appear with *scale* (corpus > window -> truncation) and *placement* (a fact the model doesn't attend to -> missed). A small-corpus demo can't observe either; the decision is about where the corpus is headed, not where the pilot sits.</details>

**Q4.** Why do the tests assert on token counts and retrieval, but never on whether the model's final answer was correct?
<details><summary>answer</summary>The answer text is model-, version-, and hardware-dependent (even at temperature 0 it is only stable for a fixed build) — asserting on it makes a brittle test that fails for the wrong reason. The *structural* facts are what the architecture guarantees and are checkable: the prompt overflowed (or didn't), the sweep genuinely fit its window, retrieval returned the gold doc, RAG sent fewer tokens. Test the property you control, not the model's wording.</details>
